In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
import sys

try:
    import pyreadstat
except ImportError as e:
    raise ImportError("Please install pyreadstat: pip install pyreadstat") from e

In [2]:
def wave_cols(stubs, waves=range(1, 17), prefix="r"):
    out = []
    for w in waves:
        for s in stubs:
            out.append(f"{prefix}{w}{s}".lower())
    return out

waves = range(1, 17)

age_stubs = ["agey_e", "agey_b"]
health_stubs = ["shlt"]
adl_item_stubs = ["walkra","dressa","batha","eata","beda","toilta"]
disease_stubs = ["diabe","cancre","lunge","hearte","stroke","psyche"]
geo_stubs = ["cenreg"]
mar_stubs = ["mstat"]

ltci_index_stubs = ["iadl5a", "adl6a", "cog27", "cesd", "bmi"]

desired = (
    ["hhidpn", "radage_y", "ragender","raracem","rahispan","raedyrs","raeduc","raedegrm"] +
    wave_cols(age_stubs, waves) +
    wave_cols(health_stubs, waves) +
    wave_cols(adl_item_stubs, waves) +
    wave_cols(disease_stubs, waves) +
    wave_cols(geo_stubs, waves) +
    wave_cols(mar_stubs, waves) +
    wave_cols(ltci_index_stubs, waves)
)

import pyreadstat

dta_path = "randhrs1992_2022v1.dta"

_, meta = pyreadstat.read_dta(dta_path, metadataonly=True)
available = set(meta.column_names)

usecols = [c for c in desired if c in available]
missing = [c for c in desired if c not in available]

print("Will load:", len(usecols))
print("Missing (first 50):", missing[:50])

df_wide, _ = pyreadstat.read_dta(dta_path, usecols=usecols)
print("Loaded:", df_wide.shape)

In [7]:
# meta already exists from metadataonly read
cog_vars = [c for c in meta.column_names if "cog27" in c.lower()]
print(cog_vars[:50], " ... total:", len(cog_vars))

In [ ]:
# configs
WAVES = range(1, 17)
ID_COL = "hhidpn"
STATE_MAP = {"H": 0, "M": 1, "S": 2, "D": 3}
DEMO_COLS = ["hhidpn", "ragender", "raracem", "rahispan", "raedyrs", "raeduc", "raedegrm"]
AGE_STUBS = ["agey_e", "agey_b"]        
HEALTH_STUBS = ["shlt"]                    # self-rated health
REGION_STUBS = ["cenreg"]
MARITAL_STUBS = ["mstat"]
DISEASE_STUBS = ["diabe", "cancre", "lunge", "hearte", "stroke", "psyche"]
INDEX_STUBS = ["adl6a", "iadl5a", "cog27", "cesd", "bmi"]
DEATH_COLS_CANDIDATES = ["radage_y"]

# load file metadata
def read_metadata_only(dta_path: str):
    try:
        _, meta = pyreadstat.read_dta(dta_path, metadataonly=True)
        return meta
    except TypeError:

        df_head, meta = pyreadstat.read_dta(dta_path, row_limit=1)

        if not getattr(meta, "column_names", None):
            class DummyMeta:
                column_names = list(df_head.columns)
            return DummyMeta()
        return meta


def wave_cols(stubs, waves=WAVES, prefix="r"):
    out = []
    for w in waves:
        for s in stubs:
            out.append(f"{prefix}{w}{s}".lower())
    return out

# detect cognition variants and add if any
def detect_cog27_variants(available_cols):
    pattern = re.compile(r"^r(\d{1,2}).*cog27.*$")
    by_wave = {w: [] for w in WAVES}
    for c in available_cols:
        m = pattern.match(c)
        if m:
            w = int(m.group(1))
            if w in by_wave:
                by_wave[w].append(c)

    chosen = {}
    for w in WAVES:
        exact = f"r{w}cog27"
        if exact in available_cols:
            chosen[w] = exact
            continue
        cands = by_wave[w]
        if not cands:
            continue

        cands_sorted = sorted(cands, key=lambda x: (len(x), x))
        chosen[w] = cands_sorted[0]
    return chosen

# detect nursing home and home care variables and add if any
def detect_utilization_stubs(available_cols):
    nh_keywords = ["nh", "nurs"]
    home_keywords = ["homcar", "homecar", "home", "hhc", "hosp", "hospice"]


    wave_stub = {}
    for c in available_cols:
        m = re.match(r"^r(\d{1,2})([a-z0-9_]+)$", c)
        if m:
            wave = int(m.group(1))
            stub = m.group(2)
            wave_stub[c] = (wave, stub)


    stub_counts = {}
    for c, (w, stub) in wave_stub.items():
        stub_counts[stub] = stub_counts.get(stub, 0) + 1

    def best_stub(keywords):
        candidates = []
        for stub, cnt in stub_counts.items():
            s = stub.lower()
            if any(k in s for k in keywords):
                candidates.append((cnt, stub))
        if not candidates:
            return None

        candidates.sort(key=lambda x: (-x[0], len(x[1]), x[1]))
        return candidates[0][1]

    return {
        "nh": best_stub(nh_keywords),
        "homecare": best_stub(home_keywords),
    }


def build_usecols(dta_path: str):
    """
    Create the final list of columns to extract from RAND .dta.
    Handles lowercase naming and cognition/utilization detection.
    """
    meta = read_metadata_only(dta_path)
    available = set([c.lower() for c in meta.column_names])


    desired = []
    desired += [c.lower() for c in DEMO_COLS if c.lower() in available]
    desired += [c.lower() for c in DEATH_COLS_CANDIDATES if c.lower() in available]

    desired += [c for c in wave_cols(AGE_STUBS) if c in available]
    desired += [c for c in wave_cols(HEALTH_STUBS) if c in available]
    desired += [c for c in wave_cols(REGION_STUBS) if c in available]
    desired += [c for c in wave_cols(MARITAL_STUBS) if c in available]
    desired += [c for c in wave_cols(DISEASE_STUBS) if c in available]


    non_cog = [s for s in INDEX_STUBS if s != "cog27"]
    desired += [c for c in wave_cols(non_cog) if c in available]


    cog_map = detect_cog27_variants(available)
    desired += list(cog_map.values())


    util = detect_utilization_stubs(available)
    util_stubs = []
    if util["nh"]:
        util_stubs.append(util["nh"])
    if util["homecare"]:
        util_stubs.append(util["homecare"])
    if util_stubs:
        desired += [c for c in wave_cols(util_stubs) if c in available]


    if ID_COL not in available:
        raise ValueError(f"ID column '{ID_COL}' not found in {dta_path}. "
                         f"Example columns: {list(sorted(list(available)))[0:30]}")


    seen = set()
    usecols = []
    for c in desired:
        if c not in seen:
            usecols.append(c)
            seen.add(c)

    info = {
        "available_n": len(available),
        "usecols_n": len(usecols),
        "cog_map": cog_map,
        "util_detected": util,
        "usecols": usecols,
    }
    return usecols, info


def extract_rand_skinny(dta_path: str) -> pd.DataFrame:
    usecols, info = build_usecols(dta_path)
    print(f"[extract] Total columns in file: {info['available_n']}")
    print(f"[extract] Will load columns: {info['usecols_n']}")
    print(f"[extract] Cognition columns chosen per wave (if any): {dict(list(info['cog_map'].items())[:5])} ...")
    print(f"[extract] Utilization stubs detected: {info['util_detected']}")

    df, _ = pyreadstat.read_dta(dta_path, usecols=usecols)
    df.columns = [c.lower() for c in df.columns]
    return df


def wide_to_long(df_wide: pd.DataFrame, id_col: str, cols: list, value_name: str) -> pd.DataFrame:
    if not cols:
        return pd.DataFrame(columns=[id_col, "Wave", value_name])
    out = df_wide[[id_col] + cols].melt(id_vars=[id_col], var_name="var", value_name=value_name)
    out["Wave"] = out["var"].str.extract(r"(\d+)").astype(int)
    out = out.drop(columns=["var"])
    return out


def build_series_long(df_wide: pd.DataFrame, stub: str, id_col: str = ID_COL, waves=WAVES, prefix="r") -> pd.DataFrame:
    cols = [f"{prefix}{w}{stub}".lower() for w in waves]
    cols = [c for c in cols if c in df_wide.columns]
    return wide_to_long(df_wide, id_col, cols, stub.upper())


def build_age_long(df_wide: pd.DataFrame, id_col: str = ID_COL, waves=WAVES) -> pd.DataFrame:

    age_e = build_series_long(df_wide, "agey_e", id_col, waves).rename(columns={"AGEY_E": "AGEY_E"})
    age_b = build_series_long(df_wide, "agey_b", id_col, waves).rename(columns={"AGEY_B": "AGEY_B"})
    out = age_e.merge(age_b, on=[id_col, "Wave"], how="outer")
    out["AGE"] = out["AGEY_E"].where(out["AGEY_E"].notna(), out["AGEY_B"])
    return out[[id_col, "Wave", "AGE"]]


def recode_binary_disease(x: pd.Series) -> pd.Series:
    return (
        pd.to_numeric(x, errors="coerce")
        .replace({2: 0, 3: 1, 4: 0, 5: 0}))


def add_missing_flag(df: pd.DataFrame, col: str, flag: str):
    if col in df.columns and df[col].isna().any():
        df[flag] = df[col].isna().astype(int)


def preprocess_rand_ltc_long(df_wide: pd.DataFrame) -> pd.DataFrame:
    df = df_wide.copy()
    if ID_COL not in df.columns:
        raise ValueError(f"Expected '{ID_COL}' in df_wide columns.")


    demo = df[[c for c in DEMO_COLS if c in df.columns]].drop_duplicates(ID_COL).copy()


    for c in demo.columns:
        if c != ID_COL:
            demo[c] = pd.to_numeric(demo[c], errors="coerce")


    age_long = build_age_long(df, ID_COL, WAVES)

    adl6 = build_series_long(df, "adl6a", ID_COL, WAVES)
    iadl5 = build_series_long(df, "iadl5a", ID_COL, WAVES)
    shlt = build_series_long(df, "shlt", ID_COL, WAVES)
    cesd = build_series_long(df, "cesd", ID_COL, WAVES)
    bmi = build_series_long(df, "bmi", ID_COL, WAVES)


    cog_cols = [c for c in df.columns if "cog27" in c.lower() and re.match(r"^r\d{1,2}.*", c.lower())]
    if cog_cols:
        cog_long = wide_to_long(df, ID_COL, cog_cols, "COG27_RAW")

        cog_long = cog_long.sort_values([ID_COL, "Wave"])
        cog_long["COG27_RAW"] = pd.to_numeric(cog_long["COG27_RAW"], errors="coerce")
        cog_long = (
            cog_long.groupby([ID_COL, "Wave"], as_index=False)["COG27_RAW"]
            .agg(lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else np.nan)
            .rename(columns={"COG27_RAW": "COG27"})
        )
    else:
        cog_long = pd.DataFrame(columns=[ID_COL, "Wave", "COG27"])


    dis_longs = []
    for d in DISEASE_STUBS:
        dl = build_series_long(df, d, ID_COL, WAVES)
        if dl.shape[1] == 3:
            col = d.upper()
            dl[col] = recode_binary_disease(dl[col]).fillna(0)
        dis_longs.append(dl)


    region = build_series_long(df, "cenreg", ID_COL, WAVES).rename(columns={"CENREG": "REGION"})
    marital = build_series_long(df, "mstat", ID_COL, WAVES).rename(columns={"MSTAT": "MARITAL_STATUS"})


    util_candidates = [c for c in df.columns if re.match(r"^r\d{1,2}.*", c) and any(k in c for k in ["nh", "nurs", "homcar", "homecar", "hosp"])]
    util_long = None
    if util_candidates:

        u = wide_to_long(df, ID_COL, util_candidates, "UTIL_RAW")
        u["UTIL_RAW"] = pd.to_numeric(u["UTIL_RAW"], errors="coerce")

        u["STUB"] = u["Wave"].astype(str).radd("r").map(lambda _: None)


        col_map = {}
        for c in util_candidates:
            m = re.match(r"^r(\d{1,2})([a-z0-9_]+)$", c)
            if m:
                col_map[c] = (int(m.group(1)), m.group(2))

        uu = df[[ID_COL] + util_candidates].melt(id_vars=[ID_COL], var_name="var", value_name="value")
        uu["value"] = pd.to_numeric(uu["value"], errors="coerce")
        uu["Wave"] = uu["var"].str.extract(r"(\d+)").astype(int)
        uu["STUB"] = uu["var"].map(lambda v: col_map.get(v, (None, None))[1])
        util_long = uu.drop(columns=["var"])

        util_wide = util_long.pivot_table(index=[ID_COL, "Wave"], columns="STUB", values="value", aggfunc="first").reset_index()


        util_wide = util_wide.rename(columns={c: c.upper() for c in util_wide.columns
            if isinstance(c, str) and c not in [ID_COL, "Wave"]})
    else:
        util_wide = pd.DataFrame(columns=[ID_COL, "Wave"])


    panel = age_long.copy()

    for piece in [adl6, iadl5, shlt, cesd, bmi, cog_long, region, marital]:
        panel = panel.merge(piece, on=[ID_COL, "Wave"], how="left")

    for dl in dis_longs:
        panel = panel.merge(dl, on=[ID_COL, "Wave"], how="left")

    if util_candidates:
        panel = panel.merge(util_wide, on=[ID_COL, "Wave"], how="left")

    panel = panel.merge(demo, on=ID_COL, how="left")


    for c in panel.columns:
        if c not in [ID_COL]:
            panel[c] = pd.to_numeric(panel[c], errors="coerce")


    panel = panel[panel["AGE"].notna()].copy()
    panel = panel[(panel["AGE"] >= 30) & (panel["AGE"] <= 100)].copy()
    panel = panel.sort_values([ID_COL, "Wave"]).reset_index(drop=True)


    panel["WAVES_SO_FAR"] = panel.groupby(ID_COL).cumcount() + 1


    panel["DAGE"] = panel.groupby(ID_COL)["AGE"].diff().fillna(0)


    panel["AGE_C"] = panel["AGE"] - 65.0
    panel["AGE2"]  = panel["AGE_C"] ** 2


    panel["AGE_GROUP"] = pd.cut(
        panel["AGE"],
        bins=[30, 40, 50, 60, 70, 80, 90, 100],
        labels=["30-39","40-49","50-59","60-69","70-79","80-89","90-100"],
        include_lowest=True)


    for col in ["ADL6A", "IADL5A", "COG27", "CESD", "BMI", "SHLT", "REGION", "MARITAL_STATUS"]:
        if col in panel.columns:
            add_missing_flag(panel, col, f"{col}_MISSING")


    if "ADL6A" in panel.columns:
        panel["ADL6A"] = panel["ADL6A"].fillna(0)
    if "IADL5A" in panel.columns:
        panel["IADL5A"] = panel["IADL5A"].fillna(0)


    dcols = [d.upper() for d in DISEASE_STUBS if d.upper() in panel.columns]
    if dcols:
        for c in dcols:
            panel[c] = recode_binary_disease(panel[c]).fillna(0)
        panel["DISEASE_COUNT"] = panel[dcols].sum(axis=1)
    else:
        panel["DISEASE_COUNT"] = 0


    if "radage_y" in df.columns:
        death_age = df[[ID_COL, "radage_y"]].drop_duplicates(ID_COL).rename(columns={"radage_y": "RADAGE_Y"})
        death_age["RADAGE_Y"] = pd.to_numeric(death_age["RADAGE_Y"], errors="coerce")
        panel = panel.merge(death_age, on=ID_COL, how="left")


        panel["DEAD"] = panel["RADAGE_Y"].notna() & (panel["AGE"] >= panel["RADAGE_Y"])
    else:
        panel["RADAGE_Y"] = np.nan
        panel["DEAD"] = False


    alive = ~panel["DEAD"]

    panel["STATE"] = np.select(
        [
            alive & (panel["ADL6A"] == 0) & (panel["IADL5A"] == 0),
            alive & ((panel["ADL6A"] == 0) & (panel["IADL5A"] >= 1) | (panel["ADL6A"] == 1)),
            alive & (panel["ADL6A"] >= 2),
            panel["DEAD"],
        ],
        ["H", "M", "S", "D"],
        default=np.nan
    )


    bad_state = panel["STATE"].isna()
    if bad_state.any():
        print("[warn] STATE is NaN for", int(bad_state.sum()), "rows. Dropping them.")
        print(panel.loc[bad_state, [ID_COL, "Wave", "AGE", "ADL6A", "IADL5A", "DEAD"]].head(20))

    panel = panel[~bad_state].copy()


    panel["dADL"] = panel.groupby(ID_COL)["ADL6A"].diff().fillna(0)
    panel["dIADL"] = panel.groupby(ID_COL)["IADL5A"].diff().fillna(0)
    if "COG27" in panel.columns:
        panel["dCOG"] = panel.groupby(ID_COL)["COG27"].diff().fillna(0)
    if "CESD" in panel.columns:
        panel["dCESD"] = panel.groupby(ID_COL)["CESD"].diff().fillna(0)
    if "BMI" in panel.columns:
        panel["dBMI"] = panel.groupby(ID_COL)["BMI"].diff().fillna(0)
    panel["dDisease"] = panel.groupby(ID_COL)["DISEASE_COUNT"].diff().fillna(0)

    panel["NEXT_STATE"] = panel.groupby(ID_COL)["STATE"].shift(-1)
    panel = panel[panel["NEXT_STATE"].notna()].copy()
    panel["Y_STATE"] = panel["NEXT_STATE"].map(STATE_MAP).astype("Int64")
    panel["STATE_ID"] = panel["STATE"].map(STATE_MAP).astype("Int64")
    panel["Y_DEATH"] = (panel["NEXT_STATE"] == "D").astype(int)


    STATE3_MAP = {"H": 0, "M": 1, "S": 2}
    panel["Y_STATE3"] = pd.Series(pd.NA, index=panel.index, dtype="Int64")

    survivors = panel["Y_DEATH"] == 0
    panel.loc[survivors, "Y_STATE3"] = panel.loc[survivors, "NEXT_STATE"].map(STATE3_MAP).astype("Int64")


    bad = survivors & panel["Y_STATE3"].isna()
    if bad.any():
        print("[warn] Found survivor rows with NEXT_STATE not in {H,M,S}. Example:")
        print(panel.loc[bad, [ID_COL, "Wave", "STATE", "NEXT_STATE"]].head(10))
        
    panel = panel.rename(columns=lambda c: c.upper() if isinstance(c, str) else c)

    return panel


if len(sys.argv) < 2:
    print("Usage: python preprocess_rand_ltc.py /path/to/randhrs1992_2022v1.dta")
    sys.exit(1)

dta_path = "randhrs1992_2022v1.dta"
print(f"[main] Reading: {dta_path}")

df_wide = extract_rand_skinny(dta_path)
print(f"[main] Extracted wide shape: {df_wide.shape}")
df_wide.head()

panel = preprocess_rand_ltc_long(df_wide)
print(f"[main] Final long shape: {panel.shape}")
panel.head()

out_csv = "Final_Preprocessed_RAND_LTCI_LONG.csv"
out_pkl = "Final_Preprocessed_RAND_LTCI_LONG.pkl"

panel.to_csv(out_csv, index=False)
panel.to_pickle(out_pkl)

print(f"[main] Saved: {out_csv}")
print(f"[main] Saved: {out_pkl}")
print("[main] Example columns:", panel.columns[:40].tolist())

In [ ]:
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

state_order = ["H", "M", "S", "D"]
state_labels = {
    "H": "Healthy",
    "M": "Mild disability",
    "S": "Severe disability",
    "D": "Death",
}
state_colors = {
    "H": "#4c72b0",  # blue
    "M": "#55a868",  # green
    "S": "#c44e52",  # red
    "D": "#8172b3",  # purple
}

# Keep only valid Next_State
df = panel[panel["NEXT_STATE"].isin(state_order)].copy()

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [6]:
counts = df["NEXT_STATE"].value_counts().reindex(state_order).fillna(0).astype(int)
total = counts.sum()

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)

bars = ax.bar(state_order, counts.values, width=0.6, color=[state_colors[s] for s in state_order], linewidth=0.8)

ax.set_xlabel("Target (Next Health State)", fontsize=10)
ax.set_ylabel("Number of Person-wave transitions", fontsize=10)

ymax = counts.max()
ax.set_ylim(0, ymax * 1.15) 

for i, (state, count) in enumerate(counts.items()):
    pct = 100 * count / total if total > 0 else 0.0
    ax.text(i, count + 0.02 * ymax, f"{pct:.1f}%", ha="center", va="bottom", fontsize=10)

patches = [
    mpatches.Patch(color=state_colors[s], label=f"{s} = {state_labels[s]}")
    for s in state_order
]
ax.legend(handles=patches, loc="upper right", frameon=True)

fig.tight_layout()
plt.savefig('class_dist.png', dpi=300, bbox_inches='tight')
plt.show()

In [12]:
df = df.sort_values(["HHIDPN", "AGE"]).copy()

# Compute age difference between consecutive visits per patient
df["Age_Difference"] = (
    df.groupby("HHIDPN")["AGE"]
      .diff()
)

df_age = df[df["Age_Difference"].notna() & (df["Age_Difference"] >= 0)].copy()

if df_age.empty:
    print("No non-negative Age_Difference values found.")
else:
    max_gap = min(df_age["Age_Difference"].max(), 15)
    bins = int(max_gap * 2) if max_gap > 0 else 10 

    fig, axes = plt.subplots(2, 2, figsize=(7, 5.2), sharex=True, sharey=True)
    axes = axes.ravel()

    for ax, state in zip(axes, state_order):
        vals = df_age.loc[df_age["NEXT_STATE"] == state, "Age_Difference"]
        if vals.empty:
            ax.set_visible(False)
            continue

        ax.hist(vals, bins=bins, color=state_colors[state], linewidth=0.5, alpha=0.8)
        ax.set_title(state_labels[state])

    # Global labels
    fig.text(0.5, 0.02, "Age difference between visits (years)", ha="center")
    fig.text(0.02, 0.5, "Count", va="center", rotation="vertical")

    fig.tight_layout(rect=[0.04, 0.04, 1, 0.93])
    plt.savefig('time_gaps.png', dpi=300, bbox_inches='tight')
    plt.show()